Mount Colab to drive

In [2]:
# Connect colab to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls "/content/drive/MyDrive/Project"

cif_files    testing.pt   Untitled0.ipynb  validating.pt
Dataset.csv  training.pt  Untitled1.ipynb


Install required libraries

In [4]:
# Install required libraries for crystal GNN
# Install necessary packages
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install network
!pip install torch-geometric torch-scatter torch-sparse   # PyG (requires matching PyTorch version)
!pip install pymatgen ase                                # For parsing CIFs
!pip install deepchem                                    # For MEGNet
!pip install pysr                                        # Symbolic Regression
!pip install shap                                       # SHAP explainability
!pip install scikit-learn                               # Data splitting metrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for network: filename=network-0.1-py3-none-any.whl size=3138 sha256=f2cf441cd80af3db9faacefeb603de265ed3c089123e80cecd453510ef9fca24
  Stored in directory: /root/.cache/pip/wheels/e7/5a/7a/7f15bea66afb5505b9d10cc7bd8964cb77f0ce736df5b104c8
Successfully built network
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677287 sha256=9ef900d880f1878b81383286341c7439243bd2d4ec20eb2ef728fb625a370894
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc

Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd

from pymatgen.core import Structure

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import CGConv, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

import warnings
warnings.filterwarnings("ignore")

Load Data

In [ ]:
data_df = "/content/drive/MyDrive/Project/Dataset.csv"

df = pd.read_csv(data_df)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7000, 17)


,mp_material_id,mp_formula,spacegroup,spacegroup_number,number of atoms,Band Gap,a,b,c,alpha,beta,gamma,Z,electronegativity,Topological Label,label,cif
0,mp-331,ScAl,Pm3m,221,2.0,0.0000,3.370349,3.370349,3.370349,90.000000,90.000000,90.000000,1.0,1.485,TI,1,cif_files/mp-331.cif
1,mp-698479,RbTeHOF4,Cc,9,32.0,4.8825,8.430517,8.430517,14.075609,85.886490,85.886490,34.382176,6.4,3.060,Trivial,0,cif_files/mp-698479.cif
2,mp-549237,Sr2Fe2S2OF2,I4/mmm,139,18.0,1.4677,4.095118,4.096245,18.221803,90.002343,90.003680,89.993481,3.6,2.458,Trivial,0,cif_files/mp-549237.cif
3,mp-13089,Cu2SnTe3,Imm2,44,6.0,0.0000,4.409910,6.113668,7.554224,113.869340,106.970766,90.000000,1.0,2.010,Trivial,0,cif_files/mp-13089.cif
4,mp-10732,ThTaN3,Pm-3m,221,5.0,0.0766,4.064437,4.064437,4.064437,90.000000,90.000000,90.000000,1.0,2.384,Topological,1,cif_files/mp-10732.cif


Feature selection

In [ ]:
#selection of features
df_data = df[[
    "spacegroup",
    "spacegroup_number",
    "number of atoms",
    "Band Gap",
    "a","b","c",
    "alpha","beta","gamma",
    "Z",
    "electronegativity",
    "label",
    "cif"
]].copy()

df_data.head()

,spacegroup,spacegroup_number,number of atoms,Band Gap,a,b,c,alpha,beta,gamma,Z,electronegativity,label,cif
0,Pm3m,221,2.0,0.0000,3.370349,3.370349,3.370349,90.000000,90.000000,90.000000,1.0,1.485,1,cif_files/mp-331.cif
1,Cc,9,32.0,4.8825,8.430517,8.430517,14.075609,85.886490,85.886490,34.382176,6.4,3.060,0,cif_files/mp-698479.cif
2,I4/mmm,139,18.0,1.4677,4.095118,4.096245,18.221803,90.002343,90.003680,89.993481,3.6,2.458,0,cif_files/mp-549237.cif
3,Imm2,44,6.0,0.0000,4.409910,6.113668,7.554224,113.869340,106.970766,90.000000,1.0,2.010,0,cif_files/mp-13089.cif
4,Pm-3m,221,5.0,0.0766,4.064437,4.064437,4.064437,90.000000,90.000000,90.000000,1.0,2.384,1,cif_files/mp-10732.cif


In [ ]:
# Encode categorical columns automatically

for col in df_data.select_dtypes(include="object").columns:
    if col != "cif":
        df_data[col] = df_data[col].astype("category").cat.codes

In [ ]:
#split data
train_df, temp_df = train_test_split(
    df_data,
    test_size=0.30,
    stratify=df_data["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 4900
Validation: 1050
Test: 1050


In [1]:
from pymatgen.io.cif import CifParser
import numpy as np
import torch
from torch_geometric.data import Data

# Example: build graph for a single CIF entry
entry = train_df.iloc[0]  # take first row of training data
cif_file = '/content/drive/MyDrive/Project' + entry['cif']
parser = CifParser(cif_files)
structure = parser.get_structures()[0]

# Node features: atomic number and electronegativity
atom_numbers = [site.specie.number for site in structure]
# Use Pettifor or Pauling electronegativity if available
atom_en = [site.specie.X if site.specie.X is not None else 0.0 for site in structure]
x = torch.tensor(list(zip(atom_numbers, atom_en)), dtype=torch.float)

# Positions (cartesian coordinates) for edge building
pos = torch.tensor(structure.cart_coords, dtype=torch.float)

# Compute edges by distance cutoff (e.g., 5 Å)
from torch_cluster import radius_graph
edge_index = radius_graph(pos, r=5.0, loop=False)
# Edge attributes: distances
row, col = edge_index
edge_attr = torch.norm(pos[row] - pos[col], dim=1, keepdim=True)

# Global (graph-level) features: spacegroup, number of atoms, band gap, lattice etc.
global_feat = torch.tensor([
    entry['spacegroup_number'], entry['number of atoms'], entry['Band Gap'],
    entry['a'], entry['b'], entry['c'],
    entry['alpha'], entry['beta'], entry['gamma'],
    entry['Z'], entry['electronegativity']
], dtype=torch.float).unsqueeze(0)  # shape [1, F]

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor([entry['label']]),
            u=global_feat, batch=torch.tensor([0]*x.size(0)))
print(data)


ModuleNotFoundError: No module named 'pymatgen'

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import CGConv, global_mean_pool

class CharlesCGCNN(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1, hidden_dim=64, num_classes=2):
        super().__init__()
        # Two CGConv layers (input dims -> hidden -> hidden)
        self.conv1 = CGConv(channels=(node_feat_dim, hidden_dim), dim=edge_feat_dim)
        self.conv2 = CGConv(channels=(hidden_dim, hidden_dim), dim=edge_feat_dim)
        # Final linear layer (after concatenating global features)
        # We'll assume global feature vector length is 11 (as constructed above)
        self.fc = nn.Linear(hidden_dim + 11, num_classes)

    def forward(self, data):
        x, edge_index, edge_attr, batch, u = data.x, data.edge_index, data.edge_attr, data.batch, data.u
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = F.relu(x)
        # Pool atomic features into graph-level vector
        x = global_mean_pool(x, batch)  # shape [batch_size, hidden_dim]
        # Concatenate global attributes u
        out = torch.cat([x, u], dim=1)  # shape [batch_size, hidden_dim+11]
        out = self.fc(out)
        return out

# Instantiate the model
torch.manual_seed(42)
model = CharlesCGCNN(node_feat_dim=2, edge_feat_dim=1, hidden_dim=64, num_classes=2).to('cuda')
print(model)


In [ ]:
#convert cif to crystal structures
from pymatgen.analysis.local_env import CrystalNN

crystal_nn = CrystalNN()

def structure_to_graph(cif_path):

    structure = Structure.from_file(cif_path)

    node_features = []
    edge_index = []
    edge_attr = []

    for i, site in enumerate(structure):
        node_features.append([site.specie.Z])

    for i, site in enumerate(structure):

        neighbors = crystal_nn.get_nn_info(structure, i)

        for nbr in neighbors:

            j = nbr['site_index']
            dist = nbr['weight']

            edge_index.append([i, j])
            edge_attr.append([dist])

    x = torch.tensor(node_features, dtype=torch.float)
    edge_index = torch.tensor(edge_index).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    return x, edge_index, edge_attr

In [ ]:
#Pytorch dataset class
class CrystalDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe, cif_root):

        self.df = dataframe.reset_index(drop=True)
        self.cif_root = cif_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        cif_path = os.path.join(self.cif_root, row["cif"])

        x, edge_index, edge_attr = structure_to_graph(cif_path)

        y = torch.tensor([row["label"]], dtype=torch.long)

        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=y
        )

        return data

In [ ]:
#create dataloaders
cif_root = "/content/drive/MyDrive/Project"

train_dataset = CrystalDataset(train_df, cif_root)
val_dataset = CrystalDataset(val_df, cif_root)
test_dataset = CrystalDataset(test_df, cif_root)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
#CGNN Model
class CGCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = CGConv(1, dim=32)
        self.conv2 = CGConv(32, dim=32)

        self.fc1 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, data):

        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))

        x = global_mean_pool(x, batch)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
#TRaining loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CGCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

criterion = nn.CrossEntropyLoss()

def train():

    model.train()
    total_loss = 0

    for data in train_loader:

        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = criterion(out, data.y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
#validation
def evaluate(loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for data in loader:

            data = data.to(device)

            out = model(data)

            prob = torch.softmax(out, dim=1)[:,1]

            preds.extend(prob.cpu().numpy())
            labels.extend(data.y.cpu().numpy())

    preds_binary = np.array(preds) > 0.5

    acc = accuracy_score(labels, preds_binary)
    f1 = f1_score(labels, preds_binary)
    roc = roc_auc_score(labels, preds)

    print("Accuracy:", acc)
    print("F1:", f1)
    print("ROC-AUC:", roc)

    print(classification_report(labels, preds_binary))

In [ ]:
#Train Model
def train():

    model.train()
    total_loss = 0

    for data in train_loader:

        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = criterion(out, data.y.view(-1))

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

def evaluate_model(loader, model):

    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for data in loader:

            data = data.to(device)

            outputs = model(data)

            probs = torch.softmax(outputs, dim=1)[:,1]  # probability of class 1

            preds = torch.argmax(outputs, dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(data.y.view(-1).cpu().numpy())

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc = roc_auc_score(all_labels, all_probs)

    print("Accuracy:", round(acc,3))
    print("Precision:", round(precision,3))
    print("Recall:", round(recall,3))
    print("F1-score:", round(f1,3))
    print("ROC-AUC:", round(roc,3))

    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds))

In [ ]:
print("Train Performance")
evaluate_model(train_loader, model)

In [ ]:
print("Validation Performance")
evaluate_model(val_loader, model)

In [ ]:
print("Test Performance")
evaluate_model(test_loader, model)

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

In [ ]:
Feature selection